# PSLL — Public Space as a Living Lab · Analysis pipeline

**Raw CSVs → validated tables → master dataset → JSON figures for the localhost dashboard.**

```
raw_data/<real|sample>/*.csv ──► this notebook ──► web/data/manifest.json
                                                   web/data/figures/*.json   (Plotly specs + data tables)
                                                   web/data/*.csv            (master datasets for stats)
```

Three layers (from *Data Analysis Info*):

| Layer | Source | Raw files |
|---|---|---|
| 1. Individual response | PsyToolkit, Polar | `affect.csv`, `physiology.csv`, `cognition.csv` |
| 2. Space-level behavior | 22 observation sheets | `observation_sessions.csv`, `movement_counts.csv`, `snapshots.csv`, `field_notes.csv` |
| 3. Context / exposure | Frigga M2H + field coding | `frigga_log.csv`, `experimental_windows.csv`, `env_manual.csv`, `spatial_conditions.csv`, `intervention_components.csv` |

Column definitions: `docs/DATA_DICTIONARY.md`. Run top to bottom (**Runtime → Run all** in Colab, **Run All** in VS Code).

## 0 · Configuration

In [ ]:
from pathlib import Path
import json, shutil, datetime as dt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    from scipy import stats as sps
except ImportError:
    sps = None

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---- Edit these ---------------------------------------------------------------
USE_SAMPLE_DATA = True          # True = synthetic test data in raw_data/sample; False = raw_data/real
COLAB_PROJECT_DIR = "/content/drive/MyDrive/PSLL Dataviz"   # where the project folder lives in Drive
SITES = ["Space A", "Space B"]
CONDITIONS = ["PRE", "POST"]
SNAPSHOT_MINUTES = [5, 10, 15, 20]

# One primary metric per cognitive test for the main presentation; the rest go to Backup.
PRIMARY_METRICS = {
    "restoration": "perceived_restoration",
    "panas":       "affect_balance",
    "rmssd":       "rmssd_ms",
    "hr":          "mean_hr_bpm",
    "attention":   "pvt_median_rt_ms",
    "inhibition":  "flanker_interference_ms",
}
DZ_THRESHOLD = 0.2              # |mean Δ / SD Δ| below this → "→ no clear change" in the evidence matrix
BEHAVIOR_PCT_THRESHOLD = 10     # % change in consensus counts below this → "→"
OBSERVER_CV_FLAG = 0.25         # flag snapshot cells where observers disagree more than this
# ------------------------------------------------------------------------------

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path(COLAB_PROJECT_DIR)
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_DIR / "raw_data" / ("sample" if USE_SAMPLE_DATA else "real")
OUT_DIR = PROJECT_DIR / "web" / "data"
print("Raw data :", RAW_DIR)
print("Output   :", OUT_DIR)
assert RAW_DIR.exists(), f"Missing {RAW_DIR} — check COLAB_PROJECT_DIR / USE_SAMPLE_DATA"

## 1 · Load and validate raw inputs

Every file is checked against a schema: required columns, allowed category values, numeric coercion.
Problems are collected in `QC` (shown at the end and exported to the dashboard) instead of stopping the run.

In [ ]:
QC = []   # list of {"level": "error"|"warning"|"info", "layer": ..., "message": ...}
def qc(level, layer, message):
    QC.append({"level": level, "layer": layer, "message": message})

from contextlib import contextmanager
@contextmanager
def section(name, layer="Pipeline"):
    """Run a block; if it fails (e.g. a layer has no data yet) log to QC and keep going."""
    try:
        yield
    except Exception as e:
        qc("warning", layer, f"{name} skipped — {type(e).__name__}: {e}")

YES_NO = ["Yes", "No"]
SCHEMAS = {
    # --- Behavior (Observation / Behavioral Sheet) ---
    "observation_sessions": dict(layer="Behavior",
        cols=["sheet_id","site","condition","observer_id","date","start_time","end_time",
              "sheet_version","boundary_confirmed","count_line_confirmed","zones_confirmed"],
        cats={"site": SITES, "condition": CONDITIONS, "sheet_version": ["short","full"]}),
    "movement_counts": dict(layer="Behavior",
        cols=["sheet_id","interval_start_min","interval_end_min","total_crossing","direction_1","direction_2"],
        num=["interval_start_min","interval_end_min","total_crossing","direction_1","direction_2"]),
    "snapshots": dict(layer="Behavior",
        cols=["sheet_id","snapshot_min","total_stationary","sitting","standing_leaning",
              "social_interaction","alone","pair_group","intervention_use"],
        num=["snapshot_min","total_stationary","sitting","standing_leaning",
             "social_interaction","alone","pair_group","intervention_use"]),
    "field_notes": dict(layer="Behavior", cols=["sheet_id","approx_time","event"]),
    # --- Environmental and Spatial Coding ---
    "frigga_log": dict(layer="Environment",
        cols=["timestamp","temperature_c","relative_humidity_pct"],
        num=["temperature_c","relative_humidity_pct"]),
    "experimental_windows": dict(layer="Environment",
        cols=["site","date","window","phase","condition","start_time","end_time"],
        cats={"site": SITES, "condition": CONDITIONS, "phase": ["rotation","behavior"]}),
    "env_manual": dict(layer="Environment",
        cols=["site","date","time","sun_condition","wind","precipitation","ground_condition","noise"],
        num=["sun_condition","wind","precipitation","ground_condition","noise"]),
    "spatial_conditions": dict(layer="Spatial",
        cols=["site","condition","available_seating","seats_occupied_start","shade","vegetation",
              "landmark","signage","ground_intervention","other_temporary","photo_file"],
        num=["available_seating","seats_occupied_start"],
        cats={"site": SITES, "condition": CONDITIONS, "shade": ["None","Partial","Majority"],
              "vegetation": ["None","Existing"], "landmark": YES_NO, "signage": YES_NO,
              "ground_intervention": YES_NO}),
    "intervention_components": dict(layer="Spatial",
        cols=["site","component","installed","units","notes"],
        num=["units"], cats={"site": SITES, "installed": YES_NO}),
    # --- Individual (PsyToolkit / Polar) ---
    "affect": dict(layer="Affect",
        cols=["participant_id","site","condition","positive_affect","negative_affect","perceived_restoration"],
        num=["positive_affect","negative_affect","perceived_restoration"],
        cats={"site": SITES, "condition": CONDITIONS}, ranges={"positive_affect": (1,7),
              "negative_affect": (1,7), "perceived_restoration": (1,7)}),
    "physiology": dict(layer="Physiology",
        cols=["participant_id","site","condition","rmssd_ms","mean_hr_bpm","sdnn_ms","quality_flag"],
        num=["rmssd_ms","mean_hr_bpm","sdnn_ms"],
        cats={"site": SITES, "condition": CONDITIONS, "quality_flag": ["ok","check","exclude"]},
        ranges={"mean_hr_bpm": (35, 200), "rmssd_ms": (1, 300)}),
    "cognition": dict(layer="Cognition",
        cols=["participant_id","site","condition","pvt_median_rt_ms","pvt_lapses","flanker_accuracy",
              "flanker_congruent_rt_ms","flanker_incongruent_rt_ms"],
        num=["pvt_median_rt_ms","pvt_lapses","flanker_accuracy","flanker_congruent_rt_ms","flanker_incongruent_rt_ms"],
        cats={"site": SITES, "condition": CONDITIONS}, ranges={"flanker_accuracy": (0, 1)}),
}

def load_table(name):
    s = SCHEMAS[name]
    path = RAW_DIR / f"{name}.csv"
    if not path.exists():
        qc("warning", s["layer"], f"{name}.csv not found — layer will be empty")
        return pd.DataFrame(columns=s["cols"])
    df = pd.read_csv(path, dtype=str, keep_default_na=False).apply(lambda c: c.str.strip())
    df = df.replace({"": np.nan})
    missing = [c for c in s["cols"] if c not in df.columns]
    if missing:
        qc("error", s["layer"], f"{name}.csv missing columns: {missing}")
        for c in missing: df[c] = np.nan
    if "condition" in df: df["condition"] = df["condition"].str.upper()
    for c in s.get("num", []):
        bad = df[c].notna() & pd.to_numeric(df[c], errors="coerce").isna()
        if bad.any(): qc("error", s["layer"], f"{name}.{c}: {bad.sum()} non-numeric value(s), set to blank")
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c, allowed in s.get("cats", {}).items():
        bad = df[c].notna() & ~df[c].isin(allowed)
        if bad.any(): qc("error", s["layer"], f"{name}.{c}: unexpected values {sorted(df.loc[bad, c].unique())} (allowed {allowed})")
    for c, (lo, hi) in s.get("ranges", {}).items():
        bad = df[c].notna() & ~df[c].between(lo, hi)
        if bad.any(): qc("warning", s["layer"], f"{name}.{c}: {bad.sum()} value(s) outside {lo}–{hi}")
    return df[s["cols"]]

raw = {name: load_table(name) for name in SCHEMAS}
pd.DataFrame({"rows": {k: len(v) for k, v in raw.items()}})

## 2 · Behavior layer — observer sheets → consensus

* **Consensus** = median across independent observers for each site × condition × snapshot (robust to one outlier sheet).
* Logical checks per sheet: `social + alone = total`, `sitting + standing ≤ total`, `intervention_use` blank for PRE.
* **Observer agreement**: coefficient of variation of `total_stationary` across observers per snapshot (Franz computes formal reliability).

In [ ]:
BEH_VARS = ["total_stationary","sitting","standing_leaning","social_interaction","alone","pair_group","intervention_use"]
BEH_LABELS = {"total_stationary": "Total stationary", "sitting": "Sitting", "standing_leaning": "Standing / leaning",
              "social_interaction": "Social interaction", "alone": "Alone", "pair_group": "Pair / group",
              "intervention_use": "Direct intervention use"}

sessions = raw["observation_sessions"]
snap = raw["snapshots"].merge(sessions[["sheet_id","site","condition","observer_id"]], on="sheet_id", how="left")
move = raw["movement_counts"].merge(sessions[["sheet_id","site","condition","observer_id"]], on="sheet_id", how="left")

orphans = snap[snap["site"].isna()]["sheet_id"].unique()
if len(orphans): qc("error", "Behavior", f"snapshots for unknown sheet_id(s): {list(orphans)}")
bad_min = snap[~snap["snapshot_min"].isin(SNAPSHOT_MINUTES)]
if len(bad_min): qc("error", "Behavior", f"{len(bad_min)} snapshot row(s) with snapshot_min not in {SNAPSHOT_MINUTES}")

chk = snap.dropna(subset=["social_interaction","alone"])
bad = chk[chk["social_interaction"] + chk["alone"] != chk["total_stationary"]]
for _, r in bad.iterrows():
    qc("warning", "Behavior", f"{r.sheet_id} @{int(r.snapshot_min)}min: social ({r.social_interaction:g}) + alone ({r.alone:g}) ≠ total ({r.total_stationary:g})")
chk = snap.dropna(subset=["sitting","standing_leaning"])
bad = chk[chk["sitting"] + chk["standing_leaning"] > chk["total_stationary"]]
for _, r in bad.iterrows():
    qc("warning", "Behavior", f"{r.sheet_id} @{int(r.snapshot_min)}min: sitting + standing > total")
bad = snap[(snap["condition"] == "PRE") & snap["intervention_use"].notna()]
if len(bad): qc("warning", "Behavior", f"{len(bad)} PRE snapshot(s) have intervention_use filled — ignored (POST only)")
snap.loc[snap["condition"] == "PRE", "intervention_use"] = np.nan

n_sheets = sessions.groupby(["site","condition"]).size().rename("sheets")
qc("info", "Behavior", f"{len(sessions)} observation sheets: " + ", ".join(f"{s} {c}={n}" for (s, c), n in n_sheets.items()))

keys = ["site","condition","snapshot_min"]
beh_consensus = snap.groupby(keys)[BEH_VARS].median()
spread = snap.groupby(keys)["total_stationary"].agg(observers="count", obs_min="min", obs_max="max",
                                                     obs_mean="mean", obs_sd="std")
spread["cv"] = (spread["obs_sd"] / spread["obs_mean"]).round(3)
beh_consensus = beh_consensus.join(spread).reset_index()
for _, r in beh_consensus[beh_consensus["cv"] > OBSERVER_CV_FLAG].iterrows():
    qc("warning", "Behavior", f"Observer disagreement {r.site} {r.condition} @{int(r.snapshot_min)}min (CV={r.cv:.2f}, range {r.obs_min:g}–{r.obs_max:g})")

move_consensus = (move.groupby(["site","condition","interval_start_min","interval_end_min"])["total_crossing"]
                  .agg(median="median", obs_min="min", obs_max="max", observers="count").reset_index())

# PRE vs POST summary: mean of the 4 consensus snapshots
beh_summary = (beh_consensus.groupby(["site","condition"])[BEH_VARS].mean().round(2)
               .reset_index().melt(id_vars=["site","condition"], var_name="behavior", value_name="mean_count"))
beh_consensus.head(8)

## 3 · Environmental layer — Frigga log → one summary per exposure window

Matches the *Environmental and Spatial Coding* table: mean / min–max temperature and RH for
PRE Rotation, PRE Behavior, POST Rotation, POST Behavior. Manual codes (sun, wind, …) are attached by time.

> Frigga gives **air temperature + RH only** — it does not characterize thermal comfort.

In [ ]:
CODE_LABELS = {
    "sun_condition":    {0: "overcast", 1: "mostly cloudy", 2: "partly sunny", 3: "clear/direct sun"},
    "wind":             {0: "calm", 1: "light", 2: "moderate", 3: "strong"},
    "precipitation":    {0: "none", 1: "light", 2: "moderate/heavy"},
    "ground_condition": {0: "dry", 1: "damp", 2: "wet"},
    "noise":            {0: "low", 1: "moderate", 2: "high"},
}
frigga = raw["frigga_log"].copy()
frigga["timestamp"] = pd.to_datetime(frigga["timestamp"], errors="coerce")
if frigga["timestamp"].isna().any(): qc("error", "Environment", "unparseable Frigga timestamps")
frigga = frigga.dropna(subset=["timestamp"]).sort_values("timestamp")

win = raw["experimental_windows"].copy()
win["start"] = pd.to_datetime(win["date"] + " " + win["start_time"], errors="coerce")
win["end"]   = pd.to_datetime(win["date"] + " " + win["end_time"], errors="coerce")

envm = raw["env_manual"].copy()
envm["ts"] = pd.to_datetime(envm["date"] + " " + envm["time"], errors="coerce")

rows = []
for _, w in win.iterrows():
    f = frigga[(frigga["timestamp"] >= w.start) & (frigga["timestamp"] <= w.end)]
    if f.empty: qc("warning", "Environment", f"No Frigga readings inside {w.site} {w.window} ({w.start_time}–{w.end_time})")
    m = envm[(envm["site"] == w.site) & envm["ts"].between(w.start - pd.Timedelta("30min"), w.end + pd.Timedelta("30min"))]
    row = dict(site=w.site, window=w.window, phase=w.phase, condition=w.condition,
               start=w.start_time, end=w.end_time, n_readings=len(f),
               temp_mean=f["temperature_c"].mean(), temp_min=f["temperature_c"].min(), temp_max=f["temperature_c"].max(),
               rh_mean=f["relative_humidity_pct"].mean(), rh_min=f["relative_humidity_pct"].min(), rh_max=f["relative_humidity_pct"].max())
    for c in CODE_LABELS:
        v = m[c].max()   # worst-case code within the window
        row[c] = CODE_LABELS[c].get(int(v), v) if pd.notna(v) else None
    rows.append(row)
env_windows = pd.DataFrame(rows).round(1)
env_windows

## 4 · Spatial layer — PRE vs POST configuration

In [ ]:
spatial = raw["spatial_conditions"].copy()
components = raw["intervention_components"].copy()
SPATIAL_LABELS = {"available_seating": "Available seating (units)", "seats_occupied_start": "Seats occupied at start",
                  "shade": "Shade over study area", "vegetation": "Vegetation", "landmark": "Landmark / focal element",
                  "signage": "Signage / directional cue", "ground_intervention": "Ground intervention",
                  "other_temporary": "Other temporary elements"}
spatial_long = (spatial.melt(id_vars=["site","condition"], value_vars=list(SPATIAL_LABELS), var_name="variable")
                .assign(variable=lambda d: d["variable"].map(SPATIAL_LABELS)))
spatial_table = spatial_long.pivot_table(index="variable", columns=["site","condition"], values="value",
                                         aggfunc="first", sort=False)
spatial_table

## 5 · Individual layer — affect, physiology, cognition → paired PRE/POST deltas

Derived: `affect_balance = positive − negative`, `flanker_interference_ms = incongruent RT − congruent RT`.
Physiology rows flagged `exclude` are blanked; `check` rows are kept and listed in QC.

In [ ]:
# Metric registry: label, units, layer, whether higher is the favorable direction (None = no valence claim)
METRICS = {
    "perceived_restoration":   ("Perceived restoration", "1–7", "Affect", True),
    "affect_balance":          ("Affect balance (positive − negative)", "points", "Affect", True),
    "positive_affect":         ("Positive affect", "1–7", "Affect", True),
    "negative_affect":         ("Negative affect", "1–7", "Affect", False),
    "rmssd_ms":                ("RMSSD", "ms", "Physiology", None),
    "mean_hr_bpm":             ("Mean heart rate", "bpm", "Physiology", None),
    "sdnn_ms":                 ("SDNN", "ms", "Physiology", None),
    "pvt_median_rt_ms":        ("PVT median reaction time", "ms", "Cognition", False),
    "pvt_lapses":              ("PVT lapses", "count", "Cognition", False),
    "flanker_interference_ms": ("Flanker interference", "ms", "Cognition", False),
    "flanker_accuracy":        ("Flanker accuracy", "proportion", "Cognition", True),
}
KEYS = ["participant_id","site","condition"]
aff = raw["affect"].assign(affect_balance=lambda d: d.positive_affect - d.negative_affect)
phy = raw["physiology"].copy()
for _, r in phy[phy["quality_flag"] == "check"].iterrows():
    qc("warning", "Physiology", f"{r.participant_id} {r.condition}: quality_flag=check (kept)")
phy.loc[phy["quality_flag"] == "exclude", ["rmssd_ms","mean_hr_bpm","sdnn_ms"]] = np.nan
cog = raw["cognition"].assign(flanker_interference_ms=lambda d: d.flanker_incongruent_rt_ms - d.flanker_congruent_rt_ms)

individual_long = aff.merge(phy, on=KEYS, how="outer").merge(cog, on=KEYS, how="outer")
dups = individual_long.duplicated(KEYS, keep=False)
if dups.any(): qc("error", "Individual", f"{dups.sum()} duplicated participant×condition rows")

if individual_long.empty:
    qc("warning", "Individual", "No affect / physiology / cognition rows yet — individual figures skipped")
    wide = pd.DataFrame(index=pd.MultiIndex.from_tuples([], names=["participant_id","site"]),
                        columns=pd.MultiIndex.from_product([list(METRICS), CONDITIONS]), dtype=float)
else:
    wide = individual_long.pivot_table(index=["participant_id","site"], columns="condition",
                                       values=list(METRICS), dropna=False)
deltas = pd.DataFrame({m: (wide[(m, "POST")] - wide[(m, "PRE")]) if (m, "POST") in wide and (m, "PRE") in wide
                       else np.nan for m in METRICS}, index=wide.index).reset_index()
incomplete = wide[wide.isna().any(axis=1)].index.get_level_values(0).tolist()
if incomplete: qc("info", "Individual", f"{len(incomplete)} participant(s) missing a PRE or POST value on ≥1 metric")

def paired_stats(metric, site):
    d = deltas.loc[deltas.site == site, metric].dropna().astype(float)
    n = len(d)
    if n < 2: return dict(n=n)
    sd = d.std(ddof=1); mean = d.mean()
    tcrit = sps.t.ppf(0.975, n - 1) if sps else 1.96
    return dict(n=n, mean_delta=mean, sd_delta=sd, dz=mean / sd if sd else np.nan,
                ci_low=mean - tcrit * sd / np.sqrt(n), ci_high=mean + tcrit * sd / np.sqrt(n))

paired = pd.DataFrame([dict(metric=m, label=METRICS[m][0], site=s, **paired_stats(m, s))
                       for m in METRICS for s in SITES]).round(3)
paired

## 6 · Multimodal evidence matrix

Direction of the PRE→POST change per evidence stream and space. Arrows describe the **metric** (↑ = value went up);
`favorable` is only stated where the doc gives a clear direction (e.g. lower RT). Physiology carries no valence.

In [ ]:
def arrow_from_dz(dz):
    if pd.isna(dz): return "–"
    return "→" if abs(dz) < DZ_THRESHOLD else ("↑" if dz > 0 else "↓")

def favorability(arrow, higher_is_better):
    if arrow in ("→", "–") or higher_is_better is None: return None
    return "favorable" if (arrow == "↑") == higher_is_better else "unfavorable"

matrix_rows = []
for row_label, key in [("Affect — restoration", "restoration"), ("Affect — PANAS", "panas"),
                       ("Physiology — RMSSD", "rmssd"), ("Physiology — HR", "hr"),
                       ("Cognition — attention", "attention"), ("Cognition — inhibition", "inhibition")]:
    m = PRIMARY_METRICS[key]; label, units, layer, hib = METRICS[m]
    cells = {}
    for s in SITES:
        st = paired_stats(m, s); a = arrow_from_dz(st.get("dz"))
        cells[s] = dict(arrow=a, favorability=favorability(a, hib),
                        detail=(f"Δ {st['mean_delta']:+.2f} {units}, dz {st['dz']:+.2f}, n={st['n']}" if "dz" in st else "insufficient data"))
    matrix_rows.append(dict(stream=row_label, metric=label, cells=cells))

def beh_change(var, site):
    v = beh_summary[(beh_summary.site == site) & (beh_summary.behavior == var)].set_index("condition")["mean_count"]
    pre, post = v.get("PRE", np.nan), v.get("POST", np.nan)
    if pd.isna(pre) or pd.isna(post) or pre == 0: return dict(arrow="–", favorability=None, detail="insufficient data")
    pct = 100 * (post - pre) / pre
    a = "→" if abs(pct) < BEHAVIOR_PCT_THRESHOLD else ("↑" if pct > 0 else "↓")
    return dict(arrow=a, favorability=None, detail=f"{pre:.1f} → {post:.1f} people ({pct:+.0f}%)")

for row_label, var in [("Behavior — occupancy", "total_stationary"), ("Behavior — social use", "social_interaction")]:
    matrix_rows.append(dict(stream=row_label, metric=BEH_LABELS[var], cells={s: beh_change(var, s) for s in SITES}))

def env_cell(site):
    if env_windows.empty: return dict(arrow="–", favorability=None, detail="no window data")
    e = env_windows[(env_windows.site == site) & (env_windows.phase == "behavior")].set_index("condition")
    if not {"PRE","POST"} <= set(e.index): return dict(arrow="–", favorability=None, detail="no window data")
    d = e.loc["POST","temp_mean"] - e.loc["PRE","temp_mean"]
    return dict(arrow="→" if abs(d) < 1 else ("↑" if d > 0 else "↓"), favorability=None,
                detail=f"air temp {d:+.1f} °C, RH {e.loc['POST','rh_mean'] - e.loc['PRE','rh_mean']:+.0f} pts (context)")
matrix_rows.append(dict(stream="Environmental context", metric="Air temperature (behavior windows)",
                        cells={s: env_cell(s) for s in SITES}))

def spatial_cell(site):
    c = components[(components.site == site) & (components.installed == "Yes")]
    return dict(arrow="+" if len(c) else "–", favorability=None,
                detail=", ".join(c["component"]) if len(c) else "no components recorded")
matrix_rows.append(dict(stream="Spatial exposure", metric="Intervention components installed",
                        cells={s: spatial_cell(s) for s in SITES}))

pd.DataFrame([{"stream": r["stream"], **{s: r["cells"][s]["arrow"] for s in SITES}} for r in matrix_rows])

## 7 · Figures

Each figure is a Plotly spec + a plain data table (the dashboard's *Show data* view).
Traces carry `meta.color` (a palette **role**, not a hex) so the web frontend re-colors them for light/dark mode.

In [ ]:
PALETTE = {"series-1": "#2a78d6", "series-2": "#eb6834", "muted": "#898781", "ink": "#52514e", "grid": "#e1e0d9"}
COND_ROLE = {"PRE": "series-1", "POST": "series-2"}
SITE_ROLE = {s: f"series-{i+1}" for i, s in enumerate(SITES)}

def base_layout(fig, height=380, **kw):
    fig.update_layout(**{**dict(template="none", height=height, margin=dict(l=56, r=16, t=76, b=48),
                      font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=13, color=PALETTE["ink"]),
                      paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", barcornerradius=4,
                      legend=dict(orientation="h", x=0, yref="container", y=0.99, yanchor="top"), hovermode="closest"), **kw})
    fig.update_xaxes(showgrid=False, linecolor="#c3c2b7", ticks="")
    fig.update_yaxes(gridcolor=PALETTE["grid"], zeroline=False, linecolor="#c3c2b7")
    return fig

def table(df):
    df = df.copy()
    return {"columns": [str(c) for c in df.columns], "rows": df.astype(object).where(df.notna(), None).values.tolist()}

FIGS = []
def register(fid, stage, layer, title, subtitle, fig=None, data_table=None, kind="plotly",
             backup=False, exploratory=False, notes=None, payload=None):
    FIGS.append(dict(id=fid, stage=stage, layer=layer, title=title, subtitle=subtitle, kind=kind,
                     backup=backup, exploratory=exploratory, notes=notes or [],
                     figure=fig.to_plotly_json() if fig is not None else None,
                     table=data_table, payload=payload))
    return fig

In [ ]:
def paired_fig(metric):
    label, units, layer, hib = METRICS[metric]
    fig = make_subplots(1, len(SITES), shared_yaxes=True, subplot_titles=SITES, horizontal_spacing=0.06)
    for i, s in enumerate(SITES, 1):
        sub = wide.xs(s, level="site")
        for pid, r in sub.iterrows():
            fig.add_trace(go.Scatter(x=CONDITIONS, y=[r.get((metric, c)) for c in CONDITIONS], mode="lines",
                line=dict(width=1, color=PALETTE["muted"]), opacity=0.45, name="Participant",
                legendgroup="p", showlegend=(i == 1 and pid == sub.index[0]),
                hovertemplate=f"{pid}<br>%{{x}}: %{{y:.2f}} {units}<extra></extra>", meta={"color": "muted"}), 1, i)
        means = [sub[(metric, c)].mean() for c in CONDITIONS]
        st = paired_stats(metric, s)
        cis = [1.96 * sub[(metric, c)].std() / np.sqrt(sub[(metric, c)].count()) for c in CONDITIONS]
        fig.add_trace(go.Scatter(x=CONDITIONS, y=means, mode="lines+markers", name="Mean ± 95% CI",
            legendgroup="m", showlegend=(i == 1), line=dict(width=2, color=PALETTE["series-1"]),
            marker=dict(size=10, color=PALETTE["series-1"], line=dict(width=2, color="#fcfcfb")),
            error_y=dict(type="data", array=cis, thickness=1.5, width=0),
            hovertemplate=f"{s} %{{x}}<br>mean %{{y:.2f}} {units}<br>n={st.get('n')}<extra></extra>",
            meta={"color": "series-1"}), 1, i)
    fig.update_yaxes(title_text=f"{label} ({units})", col=1)
    base_layout(fig)
    t = (individual_long[["participant_id","site","condition",metric]]
         .pivot_table(index=["participant_id","site"], columns="condition", values=metric).reset_index())
    t["Δ (POST−PRE)"] = t["POST"] - t["PRE"]
    return fig, table(t.round(2))

STAGE1 = [("restoration", "Affect"), ("panas", "Affect"), ("rmssd", "Physiology"), ("hr", "Physiology"),
          ("attention", "Cognition"), ("inhibition", "Cognition")]
HAS_INDIVIDUAL = not individual_long.empty
for key, layer in STAGE1:
    m = PRIMARY_METRICS[key]; label, units, _, hib = METRICS[m]
    if not HAS_INDIVIDUAL: break
    with section(f"Figure {label}", layer):
        fig, t = paired_fig(m)
        direction = {True: "Higher = favorable.", False: "Lower = favorable.", None: "Direction carries no valence on its own."}[hib]
        register(f"s1_{m}", 1, layer, f"{label}: PRE → POST", f"Each grey line is one participant. {direction}", fig, t)

primary = set(PRIMARY_METRICS.values())
for m, (label, units, layer, hib) in METRICS.items():
    if m in primary or not HAS_INDIVIDUAL: continue
    with section(f"Figure {label}", layer):
        fig, t = paired_fig(m)
        register(f"backup_{m}", 1, layer, f"{label}: PRE → POST", "Secondary metric (backup).", fig, t, backup=True)

In [ ]:
# --- Behavior: PRE/POST summary bars
with section('Behavior: PRE/POST summary bars'):
    order = ["sitting","standing_leaning","social_interaction","alone","pair_group","intervention_use"]
    fig = make_subplots(1, len(SITES), shared_yaxes=True, subplot_titles=SITES, horizontal_spacing=0.06)
    for i, s in enumerate(SITES, 1):
        for c in CONDITIONS:
            d = beh_summary[(beh_summary.site == s) & (beh_summary.condition == c)].set_index("behavior").reindex(order)
            fig.add_trace(go.Bar(x=[BEH_LABELS[b] for b in order], y=d["mean_count"].tolist(), name=c,
                                 legendgroup=c, showlegend=(i == 1), marker_color=PALETTE[COND_ROLE[c]],
                                 hovertemplate=f"{s} · {c}<br>%{{x}}: %{{y:.1f}} people<extra></extra>",
                                 meta={"color": COND_ROLE[c]}), 1, i)
    base_layout(fig, height=400, bargap=0.3, bargroupgap=0.08)
    fig.update_yaxes(title_text="People (mean of 4 snapshots)", col=1)
    bt = beh_summary.pivot_table(index=["site","behavior"], columns="condition", values="mean_count").reset_index()
    bt["behavior"] = bt["behavior"].map(BEH_LABELS)
    register("s1_behavior_summary", 1, "Behavior", "Stationary behavior: PRE vs POST",
             "Consensus (median of observers) per snapshot, averaged over the 4 snapshots. Intervention use is POST-only. "
             "Categories overlap — a person can be sitting + social + group.", fig, table(bt))


# --- Behavior: occupancy through time
with section('Behavior: occupancy through time'):
    fig = make_subplots(1, len(SITES), shared_yaxes=True, subplot_titles=SITES, horizontal_spacing=0.06)
    for i, s in enumerate(SITES, 1):
        for c in CONDITIONS:
            d = beh_consensus[(beh_consensus.site == s) & (beh_consensus.condition == c)].sort_values("snapshot_min")
            fig.add_trace(go.Scatter(x=d["snapshot_min"].tolist(), y=d["total_stationary"].tolist(), name=c,
                mode="lines+markers", legendgroup=c, showlegend=(i == 1),
                line=dict(width=2, color=PALETTE[COND_ROLE[c]]), marker=dict(size=9, color=PALETTE[COND_ROLE[c]]),
                error_y=dict(type="data", symmetric=False, array=(d["obs_max"] - d["total_stationary"]).tolist(),
                             arrayminus=(d["total_stationary"] - d["obs_min"]).tolist(), thickness=1, width=4),
                customdata=d[["obs_min","obs_max","observers"]].values.tolist(),
                hovertemplate=f"{s} · {c} · %{{x}} min<br>consensus %{{y:g}} people<br>observers %{{customdata[0]:g}}–%{{customdata[1]:g}} (n=%{{customdata[2]}})<extra></extra>",
                meta={"color": COND_ROLE[c]}), 1, i)
    base_layout(fig, height=380, hovermode="x unified")
    fig.update_xaxes(tickvals=SNAPSHOT_MINUTES, title_text="Snapshot (min)")
    fig.update_yaxes(title_text="Stationary people", rangemode="tozero", col=1)
    register("s1_occupancy_timeline", 1, "Behavior", "Stationary occupancy through time",
             "Line = observer consensus (median). Whiskers = range across observers.", fig,
             table(beh_consensus[["site","condition","snapshot_min","total_stationary","obs_min","obs_max","observers","cv"]]))


# --- Behavior: movement counts (backup)
with section('Behavior: movement counts (backup)'):
    fig = make_subplots(1, len(SITES), shared_yaxes=True, subplot_titles=SITES, horizontal_spacing=0.06)
    for i, s in enumerate(SITES, 1):
        for c in CONDITIONS:
            d = move_consensus[(move_consensus.site == s) & (move_consensus.condition == c)]
            lbl = [f"{a:02.0f}–{b:02.0f}" for a, b in zip(d.interval_start_min, d.interval_end_min)]
            fig.add_trace(go.Bar(x=lbl, y=d["median"].tolist(), name=c, legendgroup=c, showlegend=(i == 1),
                                 marker_color=PALETTE[COND_ROLE[c]], meta={"color": COND_ROLE[c]},
                                 hovertemplate=f"{s} · {c}<br>%{{x}} min: %{{y:g}} crossings<extra></extra>"), 1, i)
    base_layout(fig, bargap=0.3, bargroupgap=0.08)
    fig.update_yaxes(title_text="People crossing count line", col=1)
    register("backup_movement", 1, "Behavior", "People moving: count-line crossings per 5 min",
             "Median across observers.", fig, table(move_consensus), backup=True)

In [ ]:
# --- Environment by window
with section('Environment by window'):
    env_unique = env_windows.drop_duplicates(subset=["window","start","end"]).copy()
    env_unique["order"] = pd.to_datetime(env_unique["start"], format="%H:%M")
    env_unique = env_unique.sort_values("order")
    for var, unit, title in [("temp", "°C", "Air temperature"), ("rh", "% RH", "Relative humidity")]:
        fig = go.Figure(go.Scatter(x=env_unique["window"].tolist(), y=env_unique[f"{var}_mean"].tolist(),
            mode="markers", marker=dict(size=12, color=PALETTE["series-1"]), meta={"color": "series-1"}, name=title,
            error_y=dict(type="data", symmetric=False, thickness=1.5, width=6,
                         array=(env_unique[f"{var}_max"] - env_unique[f"{var}_mean"]).tolist(),
                         arrayminus=(env_unique[f"{var}_mean"] - env_unique[f"{var}_min"]).tolist()),
            customdata=env_unique[["start","end",f"{var}_min",f"{var}_max"]].values.tolist(),
            hovertemplate="%{x} (%{customdata[0]}–%{customdata[1]})<br>mean %{y:.1f} " + unit +
                          "<br>range %{customdata[2]}–%{customdata[3]}<extra></extra>"))
        base_layout(fig, height=340, showlegend=False)
        fig.update_yaxes(title_text=f"{title} ({unit})")
        register(f"s2_env_{var}", 2, "Environment", f"{title} by exposure window",
                 "Frigga M2H, 5-min logging. Dot = window mean, whiskers = min–max.", fig,
                 table(env_unique[["window","start","end",f"{var}_mean",f"{var}_min",f"{var}_max","n_readings"]]),
                 notes=["Air temperature and RH do not characterize thermal comfort (sun, radiant heat, wind matter)."])


# --- Environmental timeline
with section('Environmental timeline'):
    fig = make_subplots(2, 1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=["Air temperature (°C)", "Relative humidity (%)"])
    fig.add_trace(go.Scatter(x=frigga["timestamp"].dt.strftime("%Y-%m-%d %H:%M").tolist(), y=frigga["temperature_c"].tolist(),
                             mode="lines", line=dict(width=2, color=PALETTE["series-1"]), meta={"color": "series-1"},
                             name="Temperature", hovertemplate="%{x|%H:%M}: %{y:.1f} °C<extra></extra>"), 1, 1)
    fig.add_trace(go.Scatter(x=frigga["timestamp"].dt.strftime("%Y-%m-%d %H:%M").tolist(), y=frigga["relative_humidity_pct"].tolist(),
                             mode="lines", line=dict(width=2, color=PALETTE["series-1"]), meta={"color": "series-1"},
                             name="RH", hovertemplate="%{x|%H:%M}: %{y:.0f}%<extra></extra>"), 2, 1)
    for _, w in win.drop_duplicates(subset=["window","start","end"]).iterrows():
        fig.add_vrect(x0=w.start.strftime("%Y-%m-%d %H:%M"), x1=w.end.strftime("%Y-%m-%d %H:%M"),
                      fillcolor=PALETTE["grid"], opacity=0.6, line_width=0, layer="below")
        fig.add_annotation(x=(w.start + (w.end - w.start) / 2).strftime("%Y-%m-%d %H:%M"), y=1.0, yref="paper",
                           text=w.window.replace(" ", "<br>"), showarrow=False, yanchor="bottom", font=dict(size=10))
    base_layout(fig, height=460, showlegend=False, hovermode="x unified")
    fig.update_layout(margin=dict(t=90))
    register("s2_env_timeline", 2, "Environment", "Environmental timeline (09:00–17:00)",
             "Shaded bands = experimental windows.", fig,
             table(frigga.assign(timestamp=frigga.timestamp.dt.strftime("%H:%M"))))


# --- Manually coded conditions
with section('Manually coded conditions'):
    codes = env_windows[["site","window","sun_condition","wind","precipitation","ground_condition","noise"]]
    register("s2_env_codes", 2, "Environment", "Manually coded conditions per window",
             "Worst-case code observed within ±30 min of each window.", kind="table", data_table=table(codes))


# --- Spatial configuration
with section('Spatial configuration'):
    sp = spatial_table.copy()
    sp.columns = [f"{s} · {c}" for s, c in sp.columns]
    register("s2_spatial", 2, "Spatial", "Spatial configuration: baseline vs intervention",
             "Recorded once before PRE and once after installation.", kind="table",
             data_table=table(sp.reset_index().rename(columns={"variable": "Variable"})))
    register("s2_components", 2, "Spatial", "Intervention components installed (POST)", "",
             kind="table", data_table=table(components))


# --- Context flags
with section('Context flags'):
    fn = raw["field_notes"].merge(sessions[["sheet_id","site","condition","observer_id"]], on="sheet_id", how="left")
    flags = (fn.groupby(["site","condition","event"]).agg(approx_time=("approx_time","min"),
             observers_reporting=("observer_id","nunique")).reset_index())
    register("s2_context_flags", 2, "Context", "Contextual events (interpretive flags)",
             "From observer field notes; grouped by event. Not analyzed statistically.", kind="table",
             data_table=table(flags[["site","condition","approx_time","event","observers_reporting"]]))

In [ ]:
# --- Evidence matrix
with section('Evidence matrix'):
    register("s3_matrix", 3, "Synthesis", "Multimodal evidence matrix",
             f"↑/↓ = metric rose/fell (|dz| ≥ {DZ_THRESHOLD} for individual measures, ≥ {BEHAVIOR_PCT_THRESHOLD}% for behavior); "
             "→ = no clear change. Tags show favorable/unfavorable only where a direction is defined.",
             kind="matrix", payload={"sites": SITES, "rows": matrix_rows},
             data_table=table(pd.DataFrame([{"Evidence stream": r["stream"], "Metric": r["metric"],
                              **{s: f"{r['cells'][s]['arrow']}  {r['cells'][s]['detail']}" for s in SITES}} for r in matrix_rows])),
             notes=["Coincide with, not cause: PRE and POST differ in time of day, temperature and sun."])


# --- Exploratory scatterplots
with section('Exploratory scatterplots'):
    def delta_scatter(xm, ym, fid, question):
        xl, xu = METRICS[xm][0], METRICS[xm][1]; yl, yu = METRICS[ym][0], METRICS[ym][1]
        fig = go.Figure()
        notes = []
        for s in SITES:
            d = deltas[deltas.site == s].dropna(subset=[xm, ym])
            fig.add_trace(go.Scatter(x=d[xm].tolist(), y=d[ym].tolist(), mode="markers", name=s,
                marker=dict(size=10, color=PALETTE[SITE_ROLE[s]], line=dict(width=2, color="#fcfcfb")),
                text=d["participant_id"].tolist(), meta={"color": SITE_ROLE[s]},
                hovertemplate=f"%{{text}} ({s})<br>Δ {xl}: %{{x:+.2f}} {xu}<br>Δ {yl}: %{{y:+.2f}} {yu}<extra></extra>"))
            if len(d) >= 4:
                rho = d[xm].corr(d[ym], method="spearman")
                notes.append(f"{s}: Spearman ρ = {rho:+.2f} (n={len(d)})")
        fig.add_hline(y=0, line=dict(color="#c3c2b7", width=1)); fig.add_vline(x=0, line=dict(color="#c3c2b7", width=1))
        base_layout(fig, height=400)
        fig.update_xaxes(title_text=f"Δ {xl} ({xu})", showgrid=True, gridcolor=PALETTE["grid"])
        fig.update_yaxes(title_text=f"Δ {yl} ({yu})")
        register(fid, 3, "Synthesis", f"Δ {xl} × Δ {yl}", question, fig,
                 table(deltas[["participant_id","site",xm,ym]].round(2)), exploratory=True,
                 notes=notes + ["Exploratory relationship — small n, no correction for multiple comparisons."])

    delta_scatter(PRIMARY_METRICS["restoration"], PRIMARY_METRICS["rmssd"], "s3_restoration_x_rmssd",
                  "Does greater perceived restoration correspond with physiological change?")
    delta_scatter(PRIMARY_METRICS["restoration"], PRIMARY_METRICS["attention"], "s3_restoration_x_pvt",
                  "Do participants reporting greater restoration also show changes in attention?")
    delta_scatter(PRIMARY_METRICS["panas"], PRIMARY_METRICS["hr"], "s3_panas_x_hr",
                  "Does change in affect track change in physiological arousal?")

    print(f"{len(FIGS)} figures registered")

### Stage 4 · So what next?
Text cards (literature, hometown replication, citizen science, ethics) from `content/so_what_next.json`.
Edit that file, not this cell; each card becomes a dashboard card with its own reference list.

In [ ]:
SO_WHAT = None
with section('So what next? (literature)', layer="Synthesis"):
    SO_WHAT = json.loads((PROJECT_DIR / "content" / "so_what_next.json").read_text(encoding="utf-8"))
    for card in SO_WHAT["cards"]:
        refs = [{"key": k, **SO_WHAT["references"][k]} for k in card["refs"]]
        register(card["id"], SO_WHAT["stage"]["id"], card["layer"], card["title"], card["subtitle"],
                 kind="article", payload={"blocks": card["blocks"], "references": refs})
    print(f"{len(SO_WHAT['cards'])} 'So what next?' cards registered")

## 8 · Preview (optional)
Renders the main figures inline so you can check them in Colab before exporting.

In [ ]:
PREVIEW = False
if PREVIEW:
    for f in FIGS:
        if f["kind"] == "plotly" and not f["backup"]:
            go.Figure(f["figure"]).show()

## 9 · Export — JSON for the dashboard + master datasets

Writes `web/data/manifest.json`, `web/data/figures/<id>.json`, and analysis-ready CSVs.
In Colab this lands in the Drive copy of the project (synced to your laptop by Google Drive for Desktop);
a zip is also offered for download as a fallback.

In [ ]:
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return [jsonable(v) for v in o.tolist()]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if not np.isfinite(o) else round(float(o), 4)
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)): return o.isoformat()
    if o is pd.NA or o is pd.NaT: return None
    return o

STAGES = [
    dict(id=1, title="What did each measurement tell us?",
         prompt="What interpretation would you make from each data source alone?"),
    dict(id=2, title="Now look at the context",
         prompt="Does the environmental timeline or spatial configuration change your interpretation?"),
    dict(id=3, title="Put everything together",
         prompt="Where does the evidence converge or diverge? What can — and can't — we claim?"),
]
if SO_WHAT:
    STAGES.append({k: SO_WHAT["stage"][k] for k in ("id", "title", "prompt")})

fig_dir = OUT_DIR / "figures"
if fig_dir.exists(): shutil.rmtree(fig_dir)
fig_dir.mkdir(parents=True)
for f in FIGS:
    (fig_dir / f"{f['id']}.json").write_text(json.dumps(jsonable(f), ensure_ascii=False, allow_nan=False))

manifest = dict(
    project="PSLL — Public Space as a Living Lab",
    research_question="Does the spatial intervention coincide with observable changes in how people move through, occupy, and use the study area?",
    generated_at=dt.datetime.now().isoformat(timespec="seconds"),
    synthetic=USE_SAMPLE_DATA, sites=SITES, stages=STAGES,
    figures=[{k: f[k] for k in ("id","stage","layer","title","kind","backup","exploratory")} for f in FIGS],
    qc=QC,
    counts=dict(observation_sheets=len(sessions), participants=int(deltas.participant_id.nunique()),
                frigga_readings=len(frigga)),
)
(OUT_DIR / "manifest.json").write_text(json.dumps(jsonable(manifest), ensure_ascii=False, indent=2, allow_nan=False))

# Analysis-ready datasets for the stats lead
individual_long.to_csv(OUT_DIR / "master_individual_long.csv", index=False)
deltas.to_csv(OUT_DIR / "master_individual_deltas.csv", index=False)
snap.to_csv(OUT_DIR / "behavior_raw_observers.csv", index=False)
beh_consensus.to_csv(OUT_DIR / "behavior_consensus.csv", index=False)
env_windows.to_csv(OUT_DIR / "environment_windows.csv", index=False)
paired.to_csv(OUT_DIR / "paired_descriptives.csv", index=False)

print(f"Wrote {len(FIGS)} figures + manifest to {OUT_DIR}")
print(f"QC: {sum(q['level']=='error' for q in QC)} errors, {sum(q['level']=='warning' for q in QC)} warnings")
if IN_COLAB:
    from google.colab import files
    files.download(shutil.make_archive("/content/psll_web_data", "zip", OUT_DIR))
pd.DataFrame(QC)